### Exploration du prétraitement

Ce notebook construit un premier pipeline de prétraitement reproductible pour
le modèle de classification.

Les choix de feature engineering sont appliqués sans modifier le fichier CSV
brut. Le pipeline sera ajusté uniquement sur les données d'entraînement afin
d'éviter toute fuite d'information.

In [29]:
from pathlib import Path

import pandas as pd

import numpy as np

import statsmodels.api as sm

from sklearn.model_selection import train_test_split

# Définir l'emplacement du jeu de données brut
DATA_PATH = Path("../../data/raw/bank-additional.csv")

# Charger les données source
df_raw = pd.read_csv(DATA_PATH, sep=";")

# Vérifier la dimension des données brutes
df_raw.shape

(4119, 21)

In [30]:
def build_modeling_dataset(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Construit les variables explicatives et la cible du modèle."""

    # Travailler sur une copie afin de préserver les données brutes
    transformed_data = data.copy()

    # Créer l'indicateur de contact lors d'une campagne précédente
    transformed_data["previously_contacted"] = (
        transformed_data["pdays"].ne(999).astype("int8")
    )

    # Construire la cible binaire : 1 pour yes, 0 pour no
    target = transformed_data["y"].eq("yes").astype("int8")

    # Retirer la cible, duration et pdays des variables explicatives
    features = transformed_data.drop(
        columns=["y", "duration", "pdays"]
    )

    return features, target


# Construire les données utilisables pour la modélisation
X, y = build_modeling_dataset(df_raw)

# Vérifier les dimensions et le taux de souscription
print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)
print("Taux de souscription :", f"{y.mean():.2%}")

Dimensions de X : (4119, 19)
Dimensions de y : (4119,)
Taux de souscription : 10.95%


#### Séparation entraînement / test

Les données sont séparées en un jeu d'entraînement et un jeu de test.

La séparation est stratifiée selon la cible `y` afin de conserver une proportion
similaire de souscriptions dans les deux jeux. Le jeu de test ne sera utilisé
qu'à la fin pour évaluer le modèle.

In [31]:
# Séparer les données : 80 % pour l'entraînement, 20 % pour le test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

# Vérifier les dimensions et la proportion de souscriptions
split_summary = pd.DataFrame(
    {
        "jeu": ["Entraînement", "Test"],
        "nombre_lignes": [len(X_train), len(X_test)],
        "taux_souscription": [
            round(y_train.mean() * 100, 2),
            round(y_test.mean() * 100, 2),
        ],
    }
)

split_summary

,jeu,nombre_lignes,taux_souscription
0,Entraînement,3295,10.96
1,Test,824,10.92


#### Pipeline de prétraitement

Les variables numériques et catégorielles nécessitent des traitements différents.

- Les variables numériques sont imputées par leur médiane puis standardisées.
- Les variables catégorielles sont imputées par leur modalité la plus fréquente
  puis encodées avec one-hot encoding.
- Le pipeline est ajusté uniquement sur `X_train`.
- `handle_unknown="ignore"` permet de traiter sans erreur une modalité présente
  dans le jeu de test mais absente de l'entraînement.

In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identifier automatiquement les variables numériques et catégorielles
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

# Définir le traitement des variables numériques
numeric_preprocessor = Pipeline(
    steps=[
        # Remplacer d'éventuelles valeurs manquantes par la médiane
        ("imputer", SimpleImputer(strategy="median")),

        # Standardiser les échelles pour la régression logistique
        ("scaler", StandardScaler()),
    ]
)

# Définir le traitement des variables catégorielles
categorical_preprocessor = Pipeline(
    steps=[
        # Prévoir le traitement d'éventuelles valeurs manquantes
        ("imputer", SimpleImputer(strategy="most_frequent")),

        # Transformer chaque modalité en colonne binaire
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

# Combiner les deux traitements dans un unique préprocesseur
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessor, numeric_features),
        ("categorical", categorical_preprocessor, categorical_features),
    ]
)

# Ajuster le prétraitement uniquement sur les données d'entraînement
X_train_preprocessed = preprocessor.fit_transform(X_train)

# Appliquer exactement les mêmes règles au jeu de test
X_test_preprocessed = preprocessor.transform(X_test)

# Vérifier la forme des matrices finales
print("Variables numériques :", len(numeric_features))
print("Variables catégorielles :", len(categorical_features))
print("Forme après prétraitement - entraînement :", X_train_preprocessed.shape)
print("Forme après prétraitement - test :", X_test_preprocessed.shape)

Variables numériques : 9
Variables catégorielles : 10
Forme après prétraitement - entraînement : (3295, 62)
Forme après prétraitement - test : (824, 62)


In [33]:
# Afficher les colonnes créées par l'encodage qui correspondent aux modalités unknown
feature_names = preprocessor.get_feature_names_out()

unknown_encoded_features = [
    feature_name
    for feature_name in feature_names
    if "unknown" in feature_name
]

unknown_encoded_features

['categorical__job_unknown',
 'categorical__marital_unknown',
 'categorical__education_unknown',
 'categorical__default_unknown',
 'categorical__housing_unknown',
 'categorical__loan_unknown']

## Baseline : régression logistique

Une régression logistique constitue un premier modèle simple, interprétable et
adapté à une cible binaire.

Les performances sont estimées par validation croisée stratifiée sur le jeu
d'entraînement. Le jeu de test ne doit pas être utilisé à cette étape.

In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

# Construire un pipeline complet :
# chaque pli de validation croisée ajustera son propre prétraitement
baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=2_000,
                random_state=42,
            ),
        ),
    ]
)

# Créer cinq plis avec une proportion de souscriptions conservée dans chaque pli
cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

# Définir les métriques adaptées au déséquilibre de la cible
scoring_metrics = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}

# Évaluer uniquement sur le jeu d'entraînement
cross_validation_results = cross_validate(
    estimator=baseline_pipeline,
    X=X_train,
    y=y_train,
    cv=cross_validation,
    scoring=scoring_metrics,
    return_train_score=False,
)

# Résumer les résultats des cinq plis
baseline_cv_summary = pd.DataFrame(
    {
        "metrique": scoring_metrics.keys(),
        "moyenne": [
            cross_validation_results[f"test_{metric}"].mean()
            for metric in scoring_metrics
        ],
        "ecart_type": [
            cross_validation_results[f"test_{metric}"].std()
            for metric in scoring_metrics
        ],
    }
).round(3)

baseline_cv_summary

,metrique,moyenne,ecart_type
0,roc_auc,0.770,0.034
1,pr_auc,0.430,0.061
2,precision,0.590,0.053
3,recall,0.233,0.057
4,f1,0.330,0.064


#### Intrepretation

une baseline correcte et cohérente avec le déséquilibre de la cible.
- ROC-AUC = 0,770 : le modèle distingue assez bien les clients susceptibles de souscrire.
- PR-AUC = 0,430 : très au-dessus du taux de souscription de base (10,95 %), donc le modèle apporte une réelle information.
- Précision = 0,590 : quand il prédit une souscription, il a souvent raison.
- Rappel = 0,233 : il ne détecte qu’environ 23 % des clients ayant réellement souscrit.
- F1 = 0,330 : compromis encore limité entre précision et rappel.

In [35]:
# Créer une régression logistique qui compense le déséquilibre de la cible
balanced_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=2_000,
                random_state=42,
                class_weight="balanced",
            ),
        ),
    ]
)

# Évaluer ce modèle sur les mêmes plis de validation croisée
balanced_cv_results = cross_validate(
    estimator=balanced_pipeline,
    X=X_train,
    y=y_train,
    cv=cross_validation,
    scoring=scoring_metrics,
    return_train_score=False,
)

# Résumer les résultats
balanced_cv_summary = pd.DataFrame(
    {
        "metrique": scoring_metrics.keys(),
        "moyenne_modele_equilibre": [
            balanced_cv_results[f"test_{metric}"].mean()
            for metric in scoring_metrics
        ],
        "ecart_type_modele_equilibre": [
            balanced_cv_results[f"test_{metric}"].std()
            for metric in scoring_metrics
        ],
    }
).round(3)

# Comparer directement le modèle initial et le modèle équilibré
model_comparison = baseline_cv_summary.merge(
    balanced_cv_summary,
    on="metrique",
)

model_comparison

,metrique,moyenne,ecart_type,moyenne_modele_equilibre,ecart_type_modele_equilibre
0,roc_auc,0.770,0.034,0.767,0.032
1,pr_auc,0.430,0.061,0.426,0.059
2,precision,0.590,0.053,0.323,0.019
3,recall,0.233,0.057,0.621,0.054
4,f1,0.330,0.064,0.424,0.023


#### Interprétation :
- Les capacités globales de classement sont presque identiques : ROC-AUC et PR-AUC changent très peu.
- Le modèle équilibré détecte beaucoup plus de souscriptions : 62,1 % contre 23,3 %.
- En contrepartie, il génère davantage de faux positifs, donc sa précision baisse.
- Son F1 est meilleur : 0,424 contre 0,330.
Pour une campagne marketing, la décision dépendra du coût d’un appel :
- appel coûteux / équipe limitée : modèle initial, plus sélectif ;
- objectif de ne pas manquer de prospects : modèle équilibré, plus de rappel.
Pour l’instant, le modèle équilibré est le meilleur candidat, mais on ne choisit pas encore définitivement.

#### Comparaison propre les jeux complet et interprétable, avec et sans équilibrage. 

In [36]:
# Définir le jeu de variables complet
features_complete = X.columns.tolist()

# Définir le jeu interprétable avec moins de variables redondantes
features_interpretable = [
    "age",
    "campaign",
    "previous",
    "euribor3m",
    "previously_contacted",
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "poutcome",
]

feature_sets = {
    "complet": features_complete,
    "interpretable": features_interpretable,
}

In [37]:
def create_preprocessor(data: pd.DataFrame) -> ColumnTransformer:
    """Construit un préprocesseur adapté aux colonnes fournies."""

    numeric_columns = data.select_dtypes(include="number").columns.tolist()
    categorical_columns = data.select_dtypes(exclude="number").columns.tolist()

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_columns),
            ("categorical", categorical_pipeline, categorical_columns),
        ]
    )


def evaluate_logistic_regression(
    feature_columns: list[str],
    class_weight: str | None,
) -> dict:
    """Évalue une régression logistique par validation croisée."""

    # Sélectionner les colonnes pour cette expérience
    X_train_subset = X_train[feature_columns]

    # Construire un pipeline indépendant pour éviter toute fuite d'information
    model_pipeline = Pipeline(
        steps=[
            ("preprocessor", create_preprocessor(X_train_subset)),
            (
                "model",
                LogisticRegression(
                    max_iter=2_000,
                    random_state=42,
                    class_weight=class_weight,
                ),
            ),
        ]
    )

    # Évaluer uniquement sur les données d'entraînement
    scores = cross_validate(
        estimator=model_pipeline,
        X=X_train_subset,
        y=y_train,
        cv=cross_validation,
        scoring=scoring_metrics,
        return_train_score=False,
    )

    return {
        "roc_auc": scores["test_roc_auc"].mean(),
        "pr_auc": scores["test_pr_auc"].mean(),
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1": scores["test_f1"].mean(),
    }

In [38]:
experiments = []

for feature_set_name, feature_columns in feature_sets.items():
    for model_name, class_weight in [
        ("logistique_standard", None),
        ("logistique_equilibree", "balanced"),
    ]:
        scores = evaluate_logistic_regression(
            feature_columns=feature_columns,
            class_weight=class_weight,
        )

        experiments.append(
            {
                "jeu_variables": feature_set_name,
                "modele": model_name,
                **scores,
            }
        )

comparison_feature_sets = (
    pd.DataFrame(experiments)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
    .round(3)
)

comparison_feature_sets

,jeu_variables,modele,roc_auc,pr_auc,precision,recall,f1
0,complet,logistique_equilibree,0.767,0.426,0.323,0.621,0.424
1,interpretable,logistique_equilibree,0.770,0.412,0.274,0.629,0.381
2,complet,logistique_standard,0.770,0.430,0.590,0.233,0.330
3,interpretable,logistique_standard,0.772,0.419,0.605,0.197,0.293


####  On garde donc deux finalistes provisoires :

1. Complet + régression logistique équilibrée
2. Interprétable + régression logistique équilibrée

## Comparaison avec une forêt aléatoire

Une forêt aléatoire combine plusieurs arbres de décision et peut modéliser des
seuils et interactions entre variables. Elle est évaluée dans les mêmes
conditions que la régression logistique.

Le jeu de test reste inutilisé.

In [39]:
from sklearn.ensemble import RandomForestClassifier

def evaluate_random_forest(feature_columns: list[str]) -> dict:
    """Évalue une forêt aléatoire équilibrée par validation croisée."""

    # Sélectionner le jeu de variables de l'expérience
    X_train_subset = X_train[feature_columns]

    # Construire le pipeline complet
    random_forest_pipeline = Pipeline(
        steps=[
            ("preprocessor", create_preprocessor(X_train_subset)),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    min_samples_leaf=2,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    # Évaluer uniquement sur les données d'entraînement
    scores = cross_validate(
        estimator=random_forest_pipeline,
        X=X_train_subset,
        y=y_train,
        cv=cross_validation,
        scoring=scoring_metrics,
        return_train_score=False,
    )

    return {
        "roc_auc": scores["test_roc_auc"].mean(),
        "pr_auc": scores["test_pr_auc"].mean(),
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1": scores["test_f1"].mean(),
    }

In [40]:
random_forest_experiments = []

for feature_set_name, feature_columns in feature_sets.items():
    scores = evaluate_random_forest(feature_columns)

    random_forest_experiments.append(
        {
            "jeu_variables": feature_set_name,
            "modele": "foret_aleatoire_equilibree",
            **scores,
        }
    )

random_forest_comparison = (
    pd.DataFrame(random_forest_experiments)
    .round(3)
)

random_forest_comparison

,jeu_variables,modele,roc_auc,pr_auc,precision,recall,f1
0,complet,foret_aleatoire_equilibree,0.766,0.429,0.424,0.535,0.472
1,interpretable,foret_aleatoire_equilibree,0.776,0.431,0.401,0.535,0.458


À ce stade, les modèles finalistes sont :
1. Forêt aléatoire complète : meilleur équilibre global
2. Forêt aléatoire interprétable : meilleur classement global
3. Logistique équilibrée complète : meilleur rappel

#### Optimisation du seuil de décision

Le seuil par défaut d'un classifieur est 0,5 : une souscription est prédite si la
probabilité estimée est supérieure ou égale à 0,5.

Dans un contexte déséquilibré, ce seuil n'est pas nécessairement celui qui
maximise le F1-score. Le seuil est recherché uniquement à partir des données
d'entraînement, grâce à des prédictions obtenues par validation croisée.

In [41]:
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import cross_val_predict

# Construire le pipeline de la forêt aléatoire complète
best_candidate_pipeline = Pipeline(
    steps=[
        ("preprocessor", create_preprocessor(X_train[features_complete])),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

# Produire une probabilité pour chaque observation d'entraînement :
# chaque prédiction provient d'un modèle qui n'a pas vu cette observation
oof_probabilities = cross_val_predict(
    estimator=best_candidate_pipeline,
    X=X_train[features_complete],
    y=y_train,
    cv=cross_validation,
    method="predict_proba",
)[:, 1]

# Calculer précision et rappel pour tous les seuils possibles
precision_values, recall_values, thresholds = precision_recall_curve(
    y_train,
    oof_probabilities,
)

# Calculer le F1-score associé à chaque seuil
f1_values = (
    2 * precision_values[:-1] * recall_values[:-1]
    / (precision_values[:-1] + recall_values[:-1] + 1e-12)
)

# Identifier le seuil qui maximise le F1-score
best_threshold_index = f1_values.argmax()
best_threshold = thresholds[best_threshold_index]

print(f"Seuil optimisant le F1-score : {best_threshold:.3f}")
print(f"Meilleur F1-score OOF : {f1_values[best_threshold_index]:.3f}")

Seuil optimisant le F1-score : 0.565
Meilleur F1-score OOF : 0.489


In [42]:
threshold_results = []

for threshold_name, threshold in [
    ("Seuil par défaut (0.50)", 0.50),
    ("Seuil optimisé", best_threshold),
]:
    predictions = (oof_probabilities >= threshold).astype(int)

    threshold_results.append(
        {
            "seuil": threshold_name,
            "valeur_seuil": round(threshold, 3),
            "roc_auc": round(roc_auc_score(y_train, oof_probabilities), 3),
            "pr_auc": round(
                average_precision_score(y_train, oof_probabilities),
                3,
            ),
            "precision": round(precision_score(y_train, predictions), 3),
            "recall": round(recall_score(y_train, predictions), 3),
            "f1": round(f1_score(y_train, predictions), 3),
        }
    )

pd.DataFrame(threshold_results)

,seuil,valeur_seuil,roc_auc,pr_auc,precision,recall,f1
0,Seuil par défaut (0.50),0.500,0.765,0.421,0.422,0.535,0.472
1,Seuil optimisé,0.565,0.765,0.421,0.469,0.510,0.489


Pour la régression logistique équilibrée

In [43]:
# Générer des probabilités OOF pour la régression logistique équilibrée.
# Chaque prédiction est réalisée par un modèle qui n'a pas vu l'observation.
logistic_oof_probabilities = cross_val_predict(
    estimator=balanced_pipeline,
    X=X_train,
    y=y_train,
    cv=cross_validation,
    method="predict_proba",
)[:, 1]

# Calculer précision, rappel et F1 pour les différents seuils possibles.
logistic_precision, logistic_recall, logistic_thresholds = (
    precision_recall_curve(
        y_train,
        logistic_oof_probabilities,
    )
)

logistic_f1_values = (
    2 * logistic_precision[:-1] * logistic_recall[:-1]
    / (logistic_precision[:-1] + logistic_recall[:-1] + 1e-12)
)

# Identifier le seuil qui maximise le F1-score.
logistic_best_index = logistic_f1_values.argmax()
logistic_best_threshold = logistic_thresholds[logistic_best_index]

print(
    "Seuil optimal pour la régression logistique : "
    f"{logistic_best_threshold:.3f}"
)
print(
    "Meilleur F1-score OOF : "
    f"{logistic_f1_values[logistic_best_index]:.3f}"
)

Seuil optimal pour la régression logistique : 0.674
Meilleur F1-score OOF : 0.483


In [44]:
logistic_threshold_results = []

for threshold_name, threshold in [
    ("Seuil par défaut (0.50)", 0.50),
    ("Seuil optimisé", logistic_best_threshold),
]:
    predictions = (logistic_oof_probabilities >= threshold).astype(int)

    logistic_threshold_results.append(
        {
            "seuil": threshold_name,
            "valeur_seuil": round(threshold, 3),
            "roc_auc": round(
                roc_auc_score(y_train, logistic_oof_probabilities),
                3,
            ),
            "pr_auc": round(
                average_precision_score(
                    y_train,
                    logistic_oof_probabilities,
                ),
                3,
            ),
            "precision": round(
                precision_score(y_train, predictions),
                3,
            ),
            "recall": round(
                recall_score(y_train, predictions),
                3,
            ),
            "f1": round(
                f1_score(y_train, predictions),
                3,
            ),
        }
    )

pd.DataFrame(logistic_threshold_results)

,seuil,valeur_seuil,roc_auc,pr_auc,precision,recall,f1
0,Seuil par défaut (0.50),0.500,0.765,0.415,0.322,0.620,0.424
1,Seuil optimisé,0.674,0.765,0.415,0.451,0.521,0.483


   ### Seuil optimisé de la régression logistique équilibrée

   Le seuil maximisant le F1-score OOF est de 0,674. Il fait passer le F1 de
   0,424 à 0,483, un niveau proche de celui de la forêt aléatoire optimisée (0,489).

### Support Vector Machine (SVM)

Un SVM avec noyau RBF permet de modéliser des relations non linéaires entre les
variables et la souscription. Le modèle est entraîné avec des classes équilibrées
et évalué par validation croisée sur le jeu d'entraînement.

In [45]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC

# Construire un SVM non linéaire sur le jeu de variables complet.
# CalibratedClassifierCV remplace SVC(probability=True), déprécié depuis
# scikit-learn 1.9 : il transforme les scores du SVM en probabilités
# grâce à un calibrage sigmoïde (méthode de Platt).
svm_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            create_preprocessor(X_train[features_complete]),
        ),
        (
            "model",
            CalibratedClassifierCV(
                SVC(
                    kernel="rbf",
                    C=1.0,
                    gamma="scale",
                    class_weight="balanced",
                ),
                method="sigmoid",
                ensemble=False,
            ),
        ),
    ]
)

# Évaluer le SVM par validation croisée sur le jeu d'entraînement
svm_cv_results = cross_validate(
    estimator=svm_pipeline,
    X=X_train[features_complete],
    y=y_train,
    cv=cross_validation,
    scoring=scoring_metrics,
    return_train_score=False,
)

# Construire un tableau de synthèse
svm_cv_summary = pd.DataFrame(
    {
        "metrique": scoring_metrics.keys(),
        "moyenne": [
            svm_cv_results[f"test_{metric}"].mean()
            for metric in scoring_metrics
        ],
        "ecart_type": [
            svm_cv_results[f"test_{metric}"].std()
            for metric in scoring_metrics
        ],
    }
).round(3)

svm_cv_summary

,metrique,moyenne,ecart_type
0,roc_auc,0.761,0.033
1,pr_auc,0.357,0.047
2,precision,0.486,0.135
3,recall,0.050,0.016
4,f1,0.089,0.026


#### Interprétation

Le ROC-AUC (0,761) et le PR-AUC (0,357) mesurent la qualité du classement des
clients, indépendamment de tout seuil. Au seuil de 0,5, le rappel est très
faible (5 %) : le calibrage ramène les probabilités vers le taux réel de
souscription (environ 11 %), si bien que peu de clients dépassent 0,5.
Les métriques seront réévaluées au seuil optimisé.

Le PR-AUC du SVM reste inférieur à celui de la régression logistique et de la
forêt aléatoire (≈ 0,43).

In [46]:
# Probabilités out-of-fold du SVM : chaque prédiction provient
# d'un modèle qui n'a pas vu l'observation
svm_oof_probabilities = cross_val_predict(
    estimator=svm_pipeline,
    X=X_train[features_complete],
    y=y_train,
    cv=cross_validation,
    method="predict_proba",
)[:, 1]

### Comparaison des modèles et choix du seuil

Le seuil de 0,5 n'a pas la même signification pour tous les modèles :
`class_weight="balanced"` gonfle les probabilités de la régression logistique
et de la forêt aléatoire, alors que le calibrage du SVM les ramène vers le taux
réel de souscription. Les modèles sont donc comparés avec des métriques
indépendantes du seuil (ROC-AUC, PR-AUC) et au seuil optimisé propre à chacun.

Deux seuils sont déterminés à partir des probabilités OOF :

- **seuil F1** : maximise le compromis entre précision et rappel ;
- **seuil de Youden** : maximise `sensibilité + spécificité - 1`, ce qui
  favorise en général le rappel.

Les probabilités *out-of-fold* (OOF) sont obtenues par validation croisée :
chaque observation du jeu d'entraînement est prédite par un modèle entraîné
sans elle. Elles permettent de choisir un seuil de décision sans utiliser le
jeu de test.

In [47]:
from sklearn.metrics import roc_curve


def summarize_model(name: str, y_true, probabilities) -> list[dict]:
    """Calcule les métriques OOF aux seuils 0,50, F1 optimal et Youden."""

    # Seuil maximisant le F1-score
    precision_values, recall_values, pr_thresholds = precision_recall_curve(
        y_true, probabilities
    )
    f1_values = (
        2 * precision_values[:-1] * recall_values[:-1]
        / (precision_values[:-1] + recall_values[:-1] + 1e-12)
    )
    f1_threshold = pr_thresholds[f1_values.argmax()]

    # Seuil maximisant l'indice de Youden
    fpr, tpr, roc_thresholds = roc_curve(y_true, probabilities)
    youden_threshold = roc_thresholds[(tpr - fpr).argmax()]

    rows = []
    for threshold_name, threshold in [
        ("0,50", 0.50),
        ("F1 optimal", f1_threshold),
        ("Youden", youden_threshold),
    ]:
        predictions = (probabilities >= threshold).astype(int)
        rows.append(
            {
                "modele": name,
                "seuil": threshold_name,
                "valeur_seuil": round(float(threshold), 3),
                "roc_auc": round(roc_auc_score(y_true, probabilities), 3),
                "pr_auc": round(average_precision_score(y_true, probabilities), 3),
                "precision": round(precision_score(y_true, predictions, zero_division=0), 3),
                "recall": round(recall_score(y_true, predictions), 3),
                "f1": round(f1_score(y_true, predictions), 3),
            }
        )
    return rows


final_comparison = pd.DataFrame(
    summarize_model("Logistique équilibrée", y_train, logistic_oof_probabilities)
    + summarize_model("Forêt aléatoire équilibrée", y_train, oof_probabilities)
    + summarize_model("SVM RBF calibré", y_train, svm_oof_probabilities)
)

final_comparison

,modele,seuil,valeur_seuil,roc_auc,pr_auc,precision,recall,f1
0,Logistique équilibrée,"0,50",0.500,0.765,0.415,0.322,0.620,0.424
1,Logistique équilibrée,F1 optimal,0.674,0.765,0.415,0.451,0.521,0.483
2,Logistique équilibrée,Youden,0.540,0.765,0.415,0.371,0.604,0.460
3,Forêt aléatoire équilibrée,"0,50",0.500,0.765,0.421,0.422,0.535,0.472
4,Forêt aléatoire équilibrée,F1 optimal,0.565,0.765,0.421,0.469,0.510,0.489
5,Forêt aléatoire équilibrée,Youden,0.420,0.765,0.421,0.358,0.587,0.445
6,SVM RBF calibré,"0,50",0.500,0.759,0.345,0.439,0.050,0.090
7,SVM RBF calibré,F1 optimal,0.234,0.759,0.345,0.428,0.529,0.473
8,SVM RBF calibré,Youden,0.207,0.759,0.345,0.404,0.562,0.470


#### Choix du modèle final

Le SVM est écarté : malgré un F1 optimisé proche des autres modèles (0,473), son
PR-AUC est nettement inférieur (0,345) et il est le plus coûteux à entraîner.

La forêt aléatoire équilibrée obtient le meilleur PR-AUC (0,421) et le meilleur
F1 au seuil optimisé (0,489, seuil 0,565). La régression logistique équilibrée
la suit de très près (F1 0,483, seuil 0,674) ; l'écart n'est pas significatif
au regard de la variabilité entre plis.

**Décision, prise avant toute utilisation du jeu de test :**
- modèle final : forêt aléatoire équilibrée, jeu de variables complet,
  seuil 0,565 ;
- modèle de référence : régression logistique équilibrée, seuil 0,674.

Le seuil de Youden, qui privilégie le rappel, constitue une alternative si
l'objectif métier est de manquer le moins de souscripteurs possible.

## Évaluation finale sur le jeu de test

Les modèles sont réentraînés sur l'ensemble du jeu d'entraînement, puis évalués
une seule fois sur le jeu de test, avec les seuils fixés précédemment.

In [48]:
from sklearn.metrics import confusion_matrix

# Modèles et seuils fixés avant l'utilisation du jeu de test
final_models = {
    "Forêt aléatoire (modèle final)": (best_candidate_pipeline, best_threshold),
    "Logistique équilibrée (référence)": (balanced_pipeline, logistic_best_threshold),
}

test_results = []
confusion_matrices = {}

for model_name, (pipeline, threshold) in final_models.items():
    # Réentraîner sur tout le jeu d'entraînement
    pipeline.fit(X_train[features_complete], y_train)

    # Prédire le jeu de test avec le seuil fixé
    test_probabilities = pipeline.predict_proba(X_test[features_complete])[:, 1]
    test_predictions = (test_probabilities >= threshold).astype(int)

    test_results.append(
        {
            "modele": model_name,
            "seuil": round(float(threshold), 3),
            "roc_auc": round(roc_auc_score(y_test, test_probabilities), 3),
            "pr_auc": round(average_precision_score(y_test, test_probabilities), 3),
            "precision": round(precision_score(y_test, test_predictions), 3),
            "recall": round(recall_score(y_test, test_predictions), 3),
            "f1": round(f1_score(y_test, test_predictions), 3),
        }
    )

    # Conserver la matrice de confusion
    confusion_matrices[model_name] = pd.DataFrame(
        confusion_matrix(y_test, test_predictions),
        index=["Réel : non", "Réel : oui"],
        columns=["Prédit : non", "Prédit : oui"],
    )

pd.DataFrame(test_results)

,modele,seuil,roc_auc,pr_auc,precision,recall,f1
0,Forêt aléatoire (modèle final),0.565,0.76,0.419,0.429,0.433,0.431
1,Logistique équilibrée (référence),0.674,0.76,0.409,0.512,0.478,0.494


In [49]:
for model_name, matrix in confusion_matrices.items():
    print(model_name)
    display(matrix)

Forêt aléatoire (modèle final)


,Prédit : non,Prédit : oui
Réel : non,682,52
Réel : oui,51,39


Logistique équilibrée (référence)


,Prédit : non,Prédit : oui
Réel : non,693,41
Réel : oui,47,43


### Interprétation des résultats sur le jeu de test

Le ROC-AUC (0,76) et le PR-AUC (≈ 0,41-0,42) sont stables entre la validation
croisée et le test pour les deux modèles. Au seuil fixé, la régression
logistique obtient un meilleur F1 (0,494 contre 0,431), mais l'écart repose sur
quelques clients (43 souscripteurs détectés contre 39, sur 90). Les intervalles
de confiance du F1 obtenus par bootstrap se chevauchent largement
([0,347 ; 0,521] et [0,405 ; 0,581]) : les deux modèles ne sont pas
départageables.

### Choix final : la régression logistique

La forêt aléatoire avait été désignée avant le test. Les deux modèles étant
statistiquement équivalents, la régression logistique est finalement retenue
pour son interprétabilité. Ce choix est fait après consultation du jeu de test :
il repose sur l'interprétabilité, et non sur l'écart de performance observé.

In [50]:
# Estimer l'incertitude du F1 sur le test par bootstrap (1 000 rééchantillonnages)

rng = np.random.default_rng(42)
bootstrap_scores = {name: [] for name in final_models}

test_predictions_by_model = {
    name: (pipeline.predict_proba(X_test[features_complete])[:, 1] >= threshold).astype(int)
    for name, (pipeline, threshold) in final_models.items()
}

y_test_array = y_test.to_numpy()
for _ in range(1_000):
    indices = rng.integers(0, len(y_test_array), len(y_test_array))
    for name, predictions in test_predictions_by_model.items():
        bootstrap_scores[name].append(
            f1_score(y_test_array[indices], predictions[indices])
        )

pd.DataFrame(
    {
        name: {
            "f1_ic95_bas": np.percentile(scores, 2.5),
            "f1_ic95_haut": np.percentile(scores, 97.5),
        }
        for name, scores in bootstrap_scores.items()
    }
).T.round(3)

,f1_ic95_bas,f1_ic95_haut
Forêt aléatoire (modèle final),0.347,0.521
Logistique équilibrée (référence),0.405,0.581


In [51]:
# pip install optuna
import optuna
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective_logistic(trial):
    """PR-AUC moyen en validation croisée pour une régression logistique."""
    model = LogisticRegression(
        C=trial.suggest_float("C", 1e-3, 10, log=True),
        l1_ratio=trial.suggest_float("l1_ratio", 0.0, 1.0),  # 0 = L2, 1 = L1
        solver="saga",
        class_weight="balanced",
        max_iter=5_000,
        random_state=42,
    )
    pipeline = Pipeline(
        [("preprocessor", create_preprocessor(X_train)), ("model", model)]
    )
    return cross_val_score(
        pipeline, X_train, y_train, cv=cross_validation,
        scoring="average_precision",
    ).mean()


def objective_forest(trial):
    """PR-AUC moyen en validation croisée pour une forêt aléatoire."""
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 200, 800, step=100),
        max_depth=trial.suggest_int("max_depth", 3, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 20),
        max_features=trial.suggest_float("max_features", 0.1, 0.8),
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    pipeline = Pipeline(
        [("preprocessor", create_preprocessor(X_train)), ("model", model)]
    )
    return cross_val_score(
        pipeline, X_train, y_train, cv=cross_validation,
        scoring="average_precision",
    ).mean()


studies = {}
for name, objective, n_trials in [
    ("logistique", objective_logistic, 40),
    ("foret", objective_forest, 40),
]:
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(objective, n_trials=n_trials)
    studies[name] = study
    print(f"{name} : PR-AUC CV = {study.best_value:.3f} | {study.best_params}")

logistique : PR-AUC CV = 0.433 | {'C': 0.043080746450545805, 'l1_ratio': 0.04212694085665472}
foret : PR-AUC CV = 0.459 | {'n_estimators': 500, 'max_depth': 3, 'min_samples_leaf': 8, 'max_features': 0.5954139606062722}


### Optimisation des hyperparamètres (Optuna)

Les hyperparamètres sont optimisés par Optuna (40 essais, échantillonneur TPE)
en maximisant le PR-AUC moyen en validation croisée sur le jeu d'entraînement.

- **Régression logistique** : le PR-AUC passe de 0,426 à 0,433 avec une
  régularisation *elastic net* (`C ≈ 0,043`, `l1_ratio ≈ 0,04`, donc proche
  d'une pénalité L2). Une première recherche limitée à L1/L2 avait abouti à un
  score équivalent (0,437, pénalité L1, `C ≈ 0,14`) : la performance est sur un
  plateau. Le point commun des deux réglages est une régularisation plus forte
  que par défaut, cohérente avec le nombre limité de souscripteurs.
- **Forêt aléatoire** : le PR-AUC passe de 0,429 à 0,459 avec des arbres peu
  profonds (`max_depth = 3`), signe que les paramètres par défaut
  surapprenaient.

La forêt optimisée classe légèrement mieux les clients, mais la régression
logistique est retenue pour son interprétabilité : ses coefficients
s'expliquent directement en termes métier.

Pipeline final, seuil OOF et comparaison avec la version par défaut :

In [52]:
# Régression logistique avec les meilleurs hyperparamètres trouvés par Optuna
best_logistic_params = studies["logistique"].best_params

tuned_logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", create_preprocessor(X_train)),
        (
            "model",
            LogisticRegression(
                **best_logistic_params,
                solver="saga",
                class_weight="balanced",
                max_iter=5_000,
                random_state=42,
            ),
        ),
    ]
)

# Probabilités OOF pour choisir le seuil sans toucher au test
tuned_logistic_oof = cross_val_predict(
    tuned_logistic_pipeline, X_train, y_train,
    cv=cross_validation, method="predict_proba",
)[:, 1]

pd.DataFrame(
    summarize_model("Logistique par défaut", y_train, logistic_oof_probabilities)
    + summarize_model("Logistique optimisée", y_train, tuned_logistic_oof)
)

,modele,seuil,valeur_seuil,roc_auc,pr_auc,precision,recall,f1
0,Logistique par défaut,"0,50",0.500,0.765,0.415,0.322,0.620,0.424
1,Logistique par défaut,F1 optimal,0.674,0.765,0.415,0.451,0.521,0.483
2,Logistique par défaut,Youden,0.540,0.765,0.415,0.371,0.604,0.460
3,Logistique optimisée,"0,50",0.500,0.772,0.423,0.327,0.629,0.430
4,Logistique optimisée,F1 optimal,0.589,0.772,0.423,0.418,0.582,0.487
5,Logistique optimisée,Youden,0.589,0.772,0.423,0.418,0.582,0.487


### Seuil de la régression logistique optimisée

Le réglage optimisé améliore légèrement toutes les métriques OOF
(ROC-AUC 0,772 ; PR-AUC 0,423 ; F1 0,487 au seuil optimal).

Le seuil maximisant le F1 et celui maximisant l'indice de Youden coïncident
(0,589) : le choix du seuil ne dépend pas du critère retenu. À ce seuil, le
modèle détecte 58 % des souscripteurs avec une précision de 42 %.

**Modèle retenu, avant utilisation du jeu de test :** régression logistique
équilibrée, régularisation *elastic net* (`C ≈ 0,043`, `l1_ratio ≈ 0,04`),
seuil 0,589.

In [53]:
# Seuil fixé à partir des probabilités OOF du train
tuned_threshold = summarize_model(
    "Logistique optimisée", y_train, tuned_logistic_oof
)[1]["valeur_seuil"]

#  Réentraîner sur tout le jeu d'entraînement
tuned_logistic_pipeline.fit(X_train, y_train)

tuned_test_probabilities = tuned_logistic_pipeline.predict_proba(X_test)[:, 1]
tuned_test_predictions = (tuned_test_probabilities >= tuned_threshold).astype(int)

display(
    pd.DataFrame(
        [
            {
                "modele": "Logistique optimisée (finale)",
                "seuil": tuned_threshold,
                "roc_auc": round(roc_auc_score(y_test, tuned_test_probabilities), 3),
                "pr_auc": round(average_precision_score(y_test, tuned_test_probabilities), 3),
                "precision": round(precision_score(y_test, tuned_test_predictions), 3),
                "recall": round(recall_score(y_test, tuned_test_predictions), 3),
                "f1": round(f1_score(y_test, tuned_test_predictions), 3),
            }
        ]
    )
)

pd.DataFrame(
    confusion_matrix(y_test, tuned_test_predictions),
    index=["Réel : non", "Réel : oui"],
    columns=["Prédit : non", "Prédit : oui"],
)

,modele,seuil,roc_auc,pr_auc,precision,recall,f1
0,Logistique optimisée (finale),0.589,0.789,0.42,0.444,0.533,0.485


,Prédit : non,Prédit : oui
Réel : non,674,60
Réel : oui,42,48


### Évaluation finale sur le jeu de test

Les performances sur le jeu de test sont très proches des estimations OOF
(F1 : 0,485 contre 0,487 ; PR-AUC : 0,420 contre 0,423). Le modèle ne surapprend
pas et le seuil choisi sur l'entraînement se transfère correctement.

En contactant les 108 clients ciblés par le modèle (13 % du jeu de test), la
banque atteindrait 48 des 90 souscripteurs (53 %), avec une précision de 44 %,
soit environ 4 fois le taux de souscription de base (10,9 %).

Par rapport à la régression logistique non optimisée, le F1 est équivalent,
mais le modèle optimisé privilégie la détection des souscripteurs (rappel 0,533
contre 0,478), au prix de davantage d'appels.

### Coefficients du modèle final

Avec une régularisation proche de L2, 54 coefficients sur 62 restent non nuls.
Ils sont réduits vers zéro par la pénalité, donc leur ampleur est atténuée. Leur
significativité est étudiée ci-après avec un modèle non pénalisé.

In [54]:
# Coefficients du modèle final entraîné sur tout le jeu d'entraînement
coefficients = pd.DataFrame(
    {
        "variable": tuned_logistic_pipeline["preprocessor"].get_feature_names_out(),
        "coefficient": tuned_logistic_pipeline["model"].coef_[0],
    }
)
coefficients["odds_ratio"] = np.exp(coefficients["coefficient"])

(
    coefficients[coefficients["coefficient"] != 0]
    .sort_values("coefficient", key=abs, ascending=False)
    .round(3)
)

,variable,coefficient,odds_ratio
49,categorical__month_mar,0.434,1.544
3,numeric__emp.var.rate,-0.393,0.675
50,categorical__month_may,-0.390,0.677
7,numeric__nr.employed,-0.365,0.694
51,categorical__month_nov,-0.304,0.738
48,categorical__month_jun,0.297,1.346
4,numeric__cons.price.idx,0.271,1.311
59,categorical__poutcome_failure,-0.249,0.780
9,categorical__job_admin.,0.243,1.275
15,categorical__job_self-employed,-0.237,0.789


### Significativité des variables

scikit-learn ne fournit pas de p-values, et celles-ci ne sont pas valides pour
un modèle pénalisé. Une régression logistique non pénalisée est donc estimée
avec `statsmodels`, sur le jeu de variables interprétable, avec une modalité de
référence par variable catégorielle. Ce modèle sert uniquement à l'inférence :
le modèle de prédiction reste la régression logistique pénalisée.

Pour ce modèle d'inférence, certaines variables sont ajustées afin de le rendre
estimable :

- `loan` est retirée : ses clients `unknown` sont exactement ceux de `housing`,
  ce qui crée une colinéarité parfaite ;
- `default` est retirée : la modalité `yes` est quasi absente (quasi-séparation)
  et la variable n'apporte pas d'information ;
- `previously_contacted` est retirée : son information est déjà portée par
  `poutcome` ;
- la modalité `illiterate` d'`education`, trop rare, est regroupée avec
  `basic.4y`.

Ces ajustements concernent uniquement le modèle d'inférence. Le modèle de
prédiction reste inchangé.

In [55]:

# Variables du modèle d'inférence : on retire les redondances
inference_columns = [
    col for col in features_interpretable
    if col not in ["loan", "default", "previously_contacted"]
]

X_inference_source = X_train[inference_columns].copy()

# Regrouper la modalité trop rare d'education
X_inference_source["education"] = X_inference_source["education"].replace(
    {"illiterate": "basic.4y"}
)

numeric_inference = X_inference_source.select_dtypes(include="number").columns.tolist()
categorical_inference = X_inference_source.select_dtypes(exclude="number").columns.tolist()

inference_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_inference),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), categorical_inference),
    ],
    verbose_feature_names_out=False,
)

X_inference = pd.DataFrame(
    inference_preprocessor.fit_transform(X_inference_source),
    columns=inference_preprocessor.get_feature_names_out(),
    index=X_train.index,
)
X_inference = sm.add_constant(X_inference)

logit_model = sm.Logit(y_train, X_inference).fit(disp=False)

confidence_intervals = np.exp(logit_model.conf_int())
significance_table = pd.DataFrame(
    {
        "coefficient": logit_model.params,
        "odds_ratio": np.exp(logit_model.params),
        "or_ic95_bas": confidence_intervals[0],
        "or_ic95_haut": confidence_intervals[1],
        "p_value": logit_model.pvalues,
    }
).drop(index="const")
significance_table["significatif_5pct"] = significance_table["p_value"] < 0.05

significance_table.sort_values("p_value").round(3)

,coefficient,odds_ratio,or_ic95_bas,or_ic95_haut,p_value,significatif_5pct
euribor3m,-0.740,0.477,0.400,0.569,0.000,True
poutcome_success,1.704,5.496,3.262,9.260,0.000,True
month_mar,1.593,4.918,2.131,11.351,0.000,True
previous,0.317,1.373,1.145,1.647,0.001,True
month_jun,0.995,2.704,1.508,4.850,0.001,True
poutcome_nonexistent,0.969,2.637,1.484,4.683,0.001,True
month_dec,1.435,4.199,1.370,12.867,0.012,True
campaign,-0.220,0.802,0.660,0.975,0.027,True
job_management,-0.619,0.538,0.309,0.939,0.029,True
age,0.161,1.175,1.008,1.369,0.039,True


### Interprétation des effets

Les odds ratios (OR) se lisent par rapport à une modalité de référence :
avril pour `month`, admin. pour `job`, divorcé pour `marital` et échec lors de
la campagne précédente pour `poutcome`. Pour les variables numériques
(standardisées), l'OR correspond à une hausse d'un écart-type.

**Effets significatifs (p < 0,05) :**

| Variable | OR | Interprétation |
|---|---|---|
| `euribor3m` | 0,48 | Des taux d'intérêt élevés réduisent fortement les souscriptions : c'est l'effet le plus marqué |
| `poutcome_success` | 5,50 | Un client ayant souscrit lors d'une campagne précédente a environ 5,5 fois plus de chances (en odds) de souscrire qu'un client ayant refusé |
| `poutcome_nonexistent` | 2,64 | Un client jamais contacté souscrit plus qu'un client ayant déjà refusé |
| `previous` | 1,37 | Le nombre de contacts lors des campagnes passées augmente les chances de souscription |
| `month_mar`, `month_dec`, `month_jun` | 4,92 ; 4,20 ; 2,70 | Ces mois convertissent mieux qu'avril |
| `campaign` | 0,80 | Multiplier les appels durant la campagne en cours réduit les chances de souscription (lassitude) |
| `job_management` | 0,54 | Les cadres souscrivent moins que les employés administratifs |
| `age` | 1,18 | Les chances de souscription augmentent légèrement avec l'âge |

**Effets non significatifs :** éducation, prêt immobilier, type de contact,
et la plupart des catégories d'emploi. Le statut de célibataire est à la
limite de la significativité (p = 0,051).

**Cohérence avec le modèle de prédiction :** les effets majeurs (Euribor,
historique de campagne, mois, nombre d'appels) vont dans le même sens dans la
régression logistique L1, ce qui conforte leur interprétation.

**Précautions :**
- Ces résultats décrivent des associations, pas des causes. Les effets du mois
  et de l'Euribor reflètent surtout le contexte de chaque campagne (données
  2008-2010, pendant la crise financière).
- Environ 40 tests sont réalisés : au seuil de 5 %, quelques résultats
  significatifs peuvent être dus au hasard. Les effets avec p < 0,01 sont les
  plus fiables.
- Les effets de décembre et de mars reposent sur peu de contacts, d'où des
  intervalles de confiance larges.